## Data quality check before training

In [ ]:
#%pip install great_expectations
import great_expectations as gx
import great_expectations.expectations as gxe

# Convert feature table to pandas for validation
features_pd = spark.table('ml1.ml_project.customer_rfm_features').toPandas()

context = gx.get_context(mode='ephemeral')

# --- Data Source (v1.x API) ---
ds = context.data_sources.add_pandas(name='feature_data')
da = ds.add_dataframe_asset(name='rfm_features')
batch_definition = da.add_batch_definition_whole_dataframe('rfm_batch')
batch = batch_definition.get_batch(batch_parameters={'dataframe': features_pd})

# --- Expectation Suite ---
suite = context.suites.add(
    gx.ExpectationSuite(name='rfm_feature_suite')
)

# Structural expectations
suite.add_expectation(gxe.ExpectColumnToExist(column='customer_id'))
suite.add_expectation(gxe.ExpectColumnToExist(column='recency_days'))
suite.add_expectation(gxe.ExpectColumnToExist(column='frequency'))
suite.add_expectation(gxe.ExpectColumnToExist(column='monetary'))

# Value range expectations
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='recency_days', min_value=0, max_value=1825))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='frequency', min_value=1))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='monetary', min_value=0))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='customer_id'))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='recency_days'))
suite.add_expectation(gxe.ExpectColumnProportionOfUniqueValuesToBeBetween(
    column='customer_id', min_value=0.99
))

# --- Validation Definition & Run ---
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='rfm_validation',
        data=batch_definition,
        suite=suite
    )
)

results = validation_definition.run(batch_parameters={'dataframe': features_pd})

if not results.success:
    raise ValueError('Feature data quality check failed. Aborting training.')

print('Data quality checks passed. Proceeding to training.')